In [5]:
import subprocess

root = "/Users/damian/Downloads/jdownloader/ed reid - Nature's Best; New Zealand's Top 30 Songs Of All Time"

keys_file = root + "/keys.txt"
keys_details = []
with open(keys_file) as f:
    for line in f.readlines():
        parts = line.strip().split(' ', maxsplit=2)
        keys_details.append(parts)
keys_details


[['A', 'Major', 'April Sun in Cuba (128kbit_AAC).m4a.wav'],
 ['C#', 'Major', 'Beside You (128kbit_AAC).m4a.wav'],
 ['B', 'Minor', 'Counting The Beat (128kbit_AAC).m4a.wav'],
 ['F#', 'Minor', 'Dance All Around The World (128kbit_AAC).m4a.wav'],
 ['D#', 'Major', "Don't Dream It's Over (128kbit_AAC).m4a.wav"],
 ['C', 'Major', 'Drive (128kbit_AAC).m4a.wav'],
 ['E', 'Major', 'Lydia (128kbit_AAC).m4a.wav'],
 ['B', 'Major', 'Not Given Lightly (128kbit_AAC).m4a.wav'],
 ['E', 'Minor', 'She Speeds (128kbit_AAC).m4a.wav'],
 ['D',
  'Minor',
  'Shihad  - Home Again (Official Video) HD (128kbit_AAC).m4a.wav'],
 ['D#',
  'Minor',
  'Smashproof - Broken Chains (feat  Ché-Fu) [Official Video] (128kbit_AAC).m4a.wav'],
 ['D',
  'Major',
  'Split Enz - I Got You (Official Video) (128kbit_AAC).m4a.wav'],
 ['A',
  'Major',
  'Split Enz - I Hope I Never (Official Video) (128kbit_AAC).m4a.wav'],
 ['D', 'Minor', 'Split Enz - I See Red (128kbit_AAC).m4a.wav'],
 ['D',
  'Major',
  'Split Enz - Six Months In A L

In [20]:
keys_to_semis_map = {
    'C': 0,
    'C#': 1,
    'Db': 1,
    'D': 2,
    'D#': 3,
    'Eb': 3,
    'E': 4,
    'F': 5,
    'F#': 6,
    'Gb': 6,
    'G': 7,
    'G#': 8,
    'Ab': 8,
    'A': 9,
    'A#': 10,
    'Bb': 10,
    'B': 11
}
def get_pitch_shift(semi, target_semi):
    delta = target_semi - semi
    if delta > 6:
        delta -= 12
    if delta < -6:
        delta += 12
    return delta

#def get_best_pitch_shift(semis):
major_songs = [parts for parts in keys_details
                 if parts[1] == 'Major']
semis = [keys_to_semis_map[song[0]] for song in major_songs]
errors = {}
for anchor_semi in range(12):
    error = sum(abs(get_pitch_shift(semi, anchor_semi))
                for semi in semis)
    errors[anchor_semi] = error
best_semi = min(errors.items(), key=lambda kv: kv[1])[0]
errors, best_semi


({0: 25,
  1: 27,
  2: 31,
  3: 39,
  4: 41,
  5: 45,
  6: 47,
  7: 45,
  8: 41,
  9: 33,
  10: 31,
  11: 27},
 0)

In [41]:
import torchaudio
import subprocess

def write_audio(path, waveform, sr):
    if path.endswith('.wav'):
        encoding='WAV'
    elif path.endswith('.flac'):
        encoding='FLAC'
    else:
        raise ValueError('Unsupported file format - path extension must be either .wav or .flac')
    torchaudio.save_with_torchcodec(
        path,
        waveform,
        sample_rate=sr,
        encoding=encoding
    )

def pitch_shift(path, semis, output_path):
    """
    shift audio at the given path by semis semitones using librosa
    """
    cmd = [
        "rubberband",
        "--fine",
        "--formant",
        "--realtime",
        "--pitch-hq",
        "--pitch", str(semis),
        path,
        output_path
    ]
    retcode = subprocess.call(cmd)
    return retcode


In [42]:
import os
from tqdm.auto import tqdm

best_key = next(k for k,v in keys_to_semis_map.items() if v==best_semi)
print(f'best key: {best_key}')
for song in tqdm(major_songs):
    shift_semis = get_pitch_shift(keys_to_semis_map[song[0]], best_semi)
    print(f'{song[2]} in {song[0]} {song[1]} -> shift {shift_semis}')
    path = os.path.join(root, song[2])
    out_path = path + f'.shifted_to_{best_semi}.flac'
    result = pitch_shift(path, shift_semis, out_path)
    print("-> return code", result)


best key: C


  0%|          | 0/12 [00:00<?, ?it/s]

April Sun in Cuba (128kbit_AAC).m4a.wav in A Major -> shift 3


Using R3 (finer) engine
Using time ratio 1 and frequency ratio 1.18921
    
in: 9172992, out: 9172992, ratio: 1, ideal output: 9172992, error: 0
elapsed time: 28.4132 sec, in frames/sec: 322843, out frames/sec: 322843
Using R3 (finer) engine
Using time ratio 1 and frequency ratio 0.943874
0% 

-> return code 0
Beside You (128kbit_AAC).m4a.wav in C# Major -> shift -1


...

    
in: 9901056, out: 9901056, ratio: 1, ideal output: 9901056, error: 0
elapsed time: 32.7063 sec, in frames/sec: 302726, out frames/sec: 302726
Using R3 (finer) engine
Using time ratio 1 and frequency ratio 1
0% 

-> return code 0
Whaling (128kbit_AAC).m4a.wav in C Major -> shift 0


99% 

-> return code 0


    
in: 9940992, out: 9940992, ratio: 1, ideal output: 9940992, error: 0
elapsed time: 26.1402 sec, in frames/sec: 380295, out frames/sec: 380295


In [ ]:


for song in tqdm(major_songs):
    shift_semis = get_pitch_shift(keys_to_semis_map[song[0]], best_semi)
    print(f'{song[2]} in {song[0]} {song[1]} -> shift {shift_semis}')
    path = os.path.join(root, song[2])
    out_path = path + f'.shifted_to_{best_semi}.flac'
    result = pitch_shift(path, shift_semis, out_path)
    print("-> return code", result)
